In [7]:
!pip install python-docx

import os
import json
from docx import Document

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: /Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip


In [8]:
# Change this if needed
INPUT_DIR = "data/raw_docs"
OUTPUT_DIR = "data/processed"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Input Dir:", INPUT_DIR)
print("Output Dir:", OUTPUT_DIR)

Input Dir: data/raw_docs
Output Dir: data/processed


In [9]:
def parse_docx(file_path):
    doc = Document(file_path)
    lines = [p.text.strip() for p in doc.paragraphs if p.text.strip()]

    data = {
        "dataset_id": os.path.basename(file_path),
        "question": "",
        "concepts": [],
        "type": "unknown"
    }

    capture_concepts = False

    for line in lines:

        # -------- Extract question --------
        if "Prompt" in line:
            data["question"] = line

        # -------- Detect concept-based dataset --------
        if "Possible Correct Responses" in line:
            data["type"] = "concept_based"
            capture_concepts = True
            continue

        # -------- Stop capturing --------
        if capture_concepts and ("Score" in line or "Rubric" in line):
            break

        # -------- Extract concepts --------
        if capture_concepts:
            if "Needed Information" in line:
                continue

            if len(line) > 10:
                cleaned = line.replace("•", "").strip()
                data["concepts"].append(cleaned)

    return data

In [10]:
all_data = []
valid_count = 0
skipped_count = 0

files = sorted(os.listdir(INPUT_DIR))

for file in files:
    if file.endswith(".docx"):
        file_path = os.path.join(INPUT_DIR, file)

        parsed = parse_docx(file_path)
        all_data.append(parsed)

        if len(parsed["concepts"]) > 0:
            print(f"✅ {file} → concepts extracted ({len(parsed['concepts'])})")
            valid_count += 1
        else:
            print(f"⚠️ {file} → skipped (no concept section)")
            skipped_count += 1

print("\nSummary:")
print("Valid datasets:", valid_count)
print("Skipped datasets:", skipped_count)

✅ Data Set #1--ReadMeFirst.docx → concepts extracted (8)
⚠️ Data Set #10--ReadMeFirst.docx → skipped (no concept section)
⚠️ Data Set #2--ReadMeFirst.docx → skipped (no concept section)
⚠️ Data Set #3--ReadMeFirst.docx → skipped (no concept section)
⚠️ Data Set #4--ReadMeFirst.docx → skipped (no concept section)
⚠️ Data Set #5--ReadMeFirst.docx → skipped (no concept section)
⚠️ Data Set #6--ReadMeFirst.docx → skipped (no concept section)
⚠️ Data Set #7--ReadMeFirst.docx → skipped (no concept section)
⚠️ Data Set #8--ReadMeFirst.docx → skipped (no concept section)
⚠️ Data Set #9--ReadMeFirst.docx → skipped (no concept section)

Summary:
Valid datasets: 1
Skipped datasets: 9


In [12]:
# Create valid_data (ensure it exists)
valid_data = [d for d in all_data if len(d["concepts"]) > 0]

# Save all data
with open("data/processed/combined_dataset.json", "w") as f:
    json.dump(all_data, f, indent=2)

# Save only valid data
with open("data/processed/valid_dataset.json", "w") as f:
    json.dump(valid_data, f, indent=2)

print("Valid datasets:", len(valid_data))

Valid datasets: 1


In [13]:
print("Total datasets:", len(all_data))
print("Valid datasets:", len(valid_data))

for i, item in enumerate(valid_data[:2]):
    print("\n--- Dataset", i+1, "---")
    print("Question:", item["question"])
    print("Concepts:", item["concepts"][:5])

Total datasets: 10
Valid datasets: 1

--- Dataset 1 ---
Question: Prompt—Acid Rain
Concepts: ['You need to know how much vinegar was used in each container.', 'You need to know what type of vinegar was used in each container.', 'You need to know what materials to test.', 'You need to know what size/surface area of materials should be used.', 'You need to know how long each sample was rinsed in distilled water.']
